In [ ]:
# 標準
import os
from datetime import date, datetime

# サードパーティ
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error

%matplotlib inline

1.データ読み込み

In [ ]:
# 使用するカラムを指定

usecols = ["Year", "Month", "Day", "Hour", "Minute", "Temperature", "Clearsky GHI", "Cloud Type", "Dew Point", "DHI", "DNI",
            "GHI", "Relative Humidity","Pressure", "Precipitable Water", "Wind Direction", "Wind Speed"]

In [ ]:
# ファイルの読み込み＋複数年を合算


df_weather_yamanashi = pd.DataFrame()

for year in range(2016, 2021):

    file_path = f"../data/weather_yamanashi_{year}.csv"
    print(file_path)
    df = pd.read_csv(file_path, skiprows=2, usecols=usecols)

    if df_weather_yamanashi.empty:
        # print("empty")
        df_weather_yamanashi = df
    else:
        # print("not empty")
        df_weather_yamanashi =  pd.concat([df_weather_yamanashi, df])

df_weather_yamanashi.reset_index(drop = True)

2.前処理

In [ ]:
# 前処理用のdfを作成
df_weather_yamanashi_preprocessing = df_weather_yamanashi.copy()

In [ ]:
# 時刻をつくる
# 元データがtz = 0なので、utc指定。
# df_weather_yamanashi_preprocessing["utc"] = pd.to_datetime(df_weather_yamanashi_preprocessing[["Year", "Month", "Day", "Hour"]], utc = True)

df_weather_yamanashi_preprocessing["utc"] = pd.to_datetime(df_weather_yamanashi_preprocessing[["Year", "Month", "Day", "Hour"]])


In [ ]:
# time zoneの処理
# 元データはtz=0, local tz=9なので、hourを+9する。
# 23時とかは32時間とかになるからちゃんと計算する。

df_weather_yamanashi_preprocessing["jst"] = df_weather_yamanashi_preprocessing["utc"] + pd.Timedelta(hours=9)

In [ ]:
# day of yearを作成
# df_weather_yamanashi_preprocessing["day_of_year"] = df_weather_yamanashi_preprocessing["jst"].dt.dayofyear

In [ ]:
# 時刻関係を作成
df_weather_yamanashi_preprocessing["month_jst"] = df_weather_yamanashi_preprocessing["jst"].dt.month
df_weather_yamanashi_preprocessing["day_jst"] = df_weather_yamanashi_preprocessing["jst"].dt.day
df_weather_yamanashi_preprocessing["hour_jst"] = df_weather_yamanashi_preprocessing["jst"].dt.hour

In [ ]:
#日射量から、発電量を作成
# 発電量 = GHI[W/㎡] ＊ 1 [hour] * 発電効率　＊　温度補正 * パネル面積
# 発電量[Wh/m2] = GHI ＊ 1 ＊ 0.18 * (1 - 0.004 * (Temperature - 25)) * 1
# 本来は外気気温ではなく、パネル温度で補正をかける

# 発電量[kWh/m2] = GHI ＊ 1 ＊ 0.18 * (1 - 0.004 * (Temperature - 25)) * 1/1000

# [Wh]
df_weather_yamanashi_preprocessing["pv"] = df_weather_yamanashi_preprocessing["GHI"] * 1 * 0.18 * (1 - 0.004 * (df_weather_yamanashi_preprocessing["Temperature"] - 25))  * 1


3.学習/検証/テストデータ分割

In [ ]:
# 学習データ→2016年〜2018年
# 検証データ→2019年
# テストデータ→2020年

# GHIあり
feature_cols = ["pv", "GHI", "month_jst", "day_jst", "hour_jst", "Temperature"]

train_start = datetime(2016, 1, 1, 0, 0, 0)
# train_end = datetime(2018, 12, 31, 23, 59, 59)

val_start = datetime(2019, 1, 1, 0, 0, 0)
# val_end = datetime(2019, 12, 31, 23, 59, 59)

test_start = datetime(2020, 1, 1, 0, 0, 0)
# test_end = datetime(2020, 12, 31, 23, 59, 59)

df_train = df_weather_yamanashi_preprocessing[feature_cols][(df_weather_yamanashi_preprocessing["jst"] >= train_start ) 
                                           & (df_weather_yamanashi_preprocessing["jst"] < val_start)]
df_val = df_weather_yamanashi_preprocessing[feature_cols][(df_weather_yamanashi_preprocessing["jst"] >= val_start ) 
                                           & (df_weather_yamanashi_preprocessing["jst"] < test_start)]

df_test =  df_weather_yamanashi_preprocessing[feature_cols][df_weather_yamanashi_preprocessing["jst"] >= test_start]

In [ ]:
df_train

In [ ]:
df_val

4.ベースモデル構築

In [ ]:
target = "pv"

# 線形回帰のインスタンス作成
lr = LinearRegression()


In [ ]:
# 学習
X_train = df_train.drop(target, axis = 1)
Y_train =df_train[target]

lr.fit(X_train, Y_train)

In [ ]:
# 説明変数の係数
lr.coef_[0]

In [ ]:
# 切片
lr.intercept_

In [ ]:
# 検証
X_val = df_val.drop(target, axis = 1)
Y_val = df_val[target]

Y_pred_val = lr.predict(X_val)


In [ ]:
# 検証の評価指標を確認

val_mae = mean_absolute_error(Y_val, Y_pred_val)
val_rmse = mean_squared_error(Y_val, Y_pred_val)

print(f"mae:{val_mae}. rmse:{val_rmse}\n")

In [ ]:
# テスト
X_test = df_test.drop(target, axis = 1)
Y_test = df_test[target]

Y_pred_test = lr.predict(X_test)

Y_pred_val = lr.predict(X_test)

5.KPI確認

In [ ]:
# テストの評価指標を確認

test_mae = mean_absolute_error(Y_test, Y_pred_test)
test_rmse = mean_squared_error(Y_test, Y_pred_test)

print(f"mae:{test_mae}\nrmse:{test_rmse}")